# Week 74, Day 3: Documentation

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w074d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import doctest

def average(values):
    """Return the mean of a list of numbers.

    >>> average([2, 4, 9])
    5.0
    >>> average([])
    0
    """
    return sum(values) / len(values)

finder = doctest.DocTestFinder()
runner = doctest.DocTestRunner(verbose=False)
for test in finder.find(average):
    runner.run(test, out=lambda text: None)
print(runner.summarize(verbose=False))

In [ ]:
import doctest

def transfer(balances, source, target, amount):
    """Move money between two accounts in a dict of balances.

    >>> b = {"ada": 100, "bob": 5}
    >>> transfer(b, "ada", "bob", 30)
    70
    >>> b
    {'ada': 70, 'bob': 35}
    >>> transfer(b, "bob", "ada", 1000)
    Traceback (most recent call last):
        ...
    ValueError: not enough money
    """
    if amount > balances[source]:
        raise ValueError("not enough money")
    balances[source] -= amount
    balances[target] += amount
    return balances[source]

results = []
runner = doctest.DocTestRunner()
for test in doctest.DocTestFinder().find(transfer, globs={"transfer": transfer}):
    results.append(runner.run(test, out=lambda text: None))
print(results)

In [ ]:
import ast, inspect

source = '''
def area(width: float, height: float = 1.0) -> float:
    """Return the area of a rectangle.

    Args:
        width: Width in metres.
        height: Height in metres.
    """
    return width * height

def _helper():
    pass

class Shape:
    """A shape on a grid."""
'''
tree = ast.parse(source)
for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and not node.name.startswith("_"):
        doc = ast.get_docstring(node) or "(undocumented)"
        signature = f"({ast.unparse(node.args)})" if isinstance(node, ast.FunctionDef) else ""
        print(f"{node.name}{signature}: {doc.splitlines()[0]}")

ns = {}
exec(source, ns)
print(inspect.signature(ns["area"]), inspect.getdoc(ns["area"]).splitlines()[0])

> **Key idea.** A docstring is the first statement, in triple double quotes, starting with a one-line summary written as a command, then a blank line and the details: arguments, returns, errors and behaviour the signature can't show, in one style (Google or NumPy). Doctests are >>> examples with their exact output, run by doctest or pytest --doctest-modules; keep their output stable, show exceptions by their last line, and leave thorough testing to the test suite. Generate reference documentation from the code (inspect.signature and getdoc, or ast.get_docstring), and check docstring coverage and documented arguments. Sites (Sphinx, MkDocs) build docs kept with the code; every project needs a README that says what, how to install, an example, and where to go next.

### Exercise: Docstring coverage

Write `doc_coverage(source)`: count the **public** functions, classes and methods in a module (names not starting with `_`, except that `__init__` counts as public when its class is; nested functions inside functions don't count), and which of them have a docstring. Return `(documented, total, missing)`, where `missing` is a sorted list of dotted names such as `"Cart.add"`. Then `percent(source)`: the coverage as a whole-number percentage (100 for a module with nothing public).

In [ ]:
import ast

def doc_coverage(source):
    ___

def percent(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_coverage():\n    \"public names with and without docstrings\"\n    src = '''\ndef total(xs):\n    \"\"\"Return the total.\"\"\"\n    def inner():\n        pass\n    return sum(xs)\n\ndef undocumented(x):\n    return x\n\ndef _private():\n    pass\n\nclass Cart:\n    \"\"\"A shopping cart.\"\"\"\n    def __init__(self):\n        self.items = []\n    def add(self, item):\n        \"\"\"Add one item.\"\"\"\n        self.items.append(item)\n    def _check(self):\n        pass\n\nclass _Hidden:\n    def visible(self):\n        pass\n'''\n    assert doc_coverage(src) == (3, 5, [\"Cart.__init__\", \"undocumented\"])\n    assert percent(src) == 60 and percent(\"x = 1\\n\") == 100")

### Exercise: Reading a Google-style docstring

Write `parse_docstring(doc)`: split a cleaned Google-style docstring into a dict with `"summary"` (the first line), `"description"` (the text between the summary and the first section, stripped, or `""`), `"args"` (a dict of argument name to its description, from an `Args:` section; a description may continue on more-indented lines, joined with single spaces), `"returns"` (the text of `Returns:`, joined into one line, or `None`) and `"raises"` (a dict of exception name to description). A section heading is one of `Args:`, `Arguments:`, `Returns:`, `Raises:` and `Examples:` on a line of its own; the content of an `Examples:` section is ignored.

In [ ]:
import inspect, re

def parse_docstring(doc):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_google():\n    \"sections, entries and continuation lines\"\n    import inspect\n    doc = inspect.cleandoc('''Move money between two accounts.\n\n        Both accounts must belong to the same bank.\n\n        Args:\n            source: The account to take money from.\n            target: The account to pay into.\n            amount: How much to move, in pence.\n                Must be positive.\n\n        Returns:\n            The new balance of\n            the source account.\n\n        Raises:\n            ValueError: If amount isn't positive.\n            KeyError: If an account doesn't exist.\n\n        Examples:\n            >>> transfer(a, b, 5)\n    ''')\n    d = parse_docstring(doc)\n    assert d[\"summary\"] == \"Move money between two accounts.\" and d[\"description\"] == \"Both accounts must belong to the same bank.\"\n    assert d[\"args\"] == {\"source\": \"The account to take money from.\", \"target\": \"The account to pay into.\", \"amount\": \"How much to move, in pence. Must be positive.\"}\n    assert d[\"returns\"] == \"The new balance of the source account.\"\n    assert d[\"raises\"] == {\"ValueError\": \"If amount isn't positive.\", \"KeyError\": \"If an account doesn't exist.\"}\n    bare = parse_docstring(\"Return x.\")\n    assert bare == {\"summary\": \"Return x.\", \"description\": \"\", \"args\": {}, \"returns\": None, \"raises\": {}}")

### Exercise: Documented arguments that don't exist

Write `check_args(func, documented)`: compare the names in `documented` (a dict from a parsed `Args:` section) with `func`'s parameters from `inspect.signature`, ignoring `self` and `cls`. Return a sorted list of messages: `"undocumented: NAME"` for each parameter not documented, and `"no such argument: NAME"` for each documented name that isn't a parameter. For `*args` and `**kwargs`, accept the documented names `args`, `*args`, `kwargs` and `**kwargs`.

In [ ]:
import inspect

def check_args(func, documented):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_args():\n    \"docstrings that match the signature\"\n    def transfer(source, target, amount=0, *, note=None):\n        pass\n    assert check_args(transfer, {\"source\": \"\", \"target\": \"\", \"amount\": \"\", \"note\": \"\"}) == []\n    assert check_args(transfer, {\"source\": \"\", \"dest\": \"\", \"amount\": \"\"}) == [\"no such argument: dest\", \"undocumented: note\", \"undocumented: target\"]\n    class Cart:\n        def add(self, item, *extra, **options):\n            pass\n    assert check_args(Cart().add, {\"item\": \"\", \"*extra\": \"\", \"**options\": \"\"}) == []\n    assert check_args(Cart.add, {\"item\": \"\", \"extra\": \"\"}) == [\"undocumented: options\"]")

### Exercise: Examples that run

Write `slugify(title)` (lower case, runs of non-alphanumeric characters become one hyphen, no hyphens at the ends) with a docstring containing **at least three** doctest examples, including one for an empty title (which returns `""`) and one showing that `"  Hello, World!  "` gives `'hello-world'`. The checks run your doctests, and the function itself.

In [ ]:
import re

def slugify(title):
    """Make a URL-friendly slug from a title.
    """
    return title

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_doctests():\n    \"documented examples that pass\"\n    import doctest\n    tests = doctest.DocTestFinder().find(slugify, globs={\"slugify\": slugify})\n    examples = sum(len(t.examples) for t in tests)\n    runner = doctest.DocTestRunner()\n    for t in tests:\n        runner.run(t, out=lambda s: None)\n    result = runner.summarize(verbose=False)\n    assert examples >= 3, \"write at least three examples\"\n    assert result.failed == 0, \"some examples fail\"\n    doc = slugify.__doc__\n    assert \"'hello-world'\" in doc and \"''\" in doc\n    assert slugify(\"Week 74: Code quality\") == \"week-74-code-quality\" and slugify(\"--a--\") == \"a\"")

### Exercise: Reference documentation from source

Write `api_reference(source, module_name)`: Markdown reference documentation for a module's public functions and classes (in source order), generated with `ast` without running the code. Start with `f"# {module_name}"`, then the module docstring's first line (if any) after a blank line. For each public function: a blank line, `` f"## `{name}({args})`" `` (with `ast.unparse` of its arguments) and, after a blank line, its docstring's first line (or `*Undocumented.*`). For each public class: `` f"## `class {name}`" ``, its summary the same way, then for each public method (and `__init__`) a blank line, `` f"### `{name}.{method}({args})`" `` (leaving out a leading `self`), a blank line and its summary. Join with newlines and end with one.

In [ ]:
import ast

def api_reference(source, module_name):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_reference():\n    \"generated from the code\"\n    src = '''\"\"\"Small shopping helpers.\n\nMore detail here.\n\"\"\"\n\ndef total(prices, tax=0.2):\n    \"\"\"Return the total with tax.\"\"\"\n    return sum(prices) * (1 + tax)\n\ndef _secret():\n    pass\n\nclass Cart:\n    \"\"\"A shopping cart.\"\"\"\n    def __init__(self, owner):\n        self.owner = owner\n    def add(self, item, qty=1):\n        \"\"\"Add an item.\"\"\"\n'''\n    expected = \"\"\"# shop\n\nSmall shopping helpers.\n\n## `total(prices, tax=0.2)`\n\nReturn the total with tax.\n\n## `class Cart`\n\nA shopping cart.\n\n### `Cart.__init__(owner)`\n\n*Undocumented.*\n\n### `Cart.add(item, qty=1)`\n\nAdd an item.\n\"\"\"\n    assert api_reference(src, \"shop\") == expected\n    assert api_reference(\"x = 1\\n\", \"empty\") == \"# empty\\n\"")

### Exercise: Checking the summary line

Write `summary_problems(doc)`: a list of problems with a raw docstring (as written, before cleaning), in this order: `"empty"` (and nothing else) if it has no text; `"starts with a blank line"` if the text starts with a newline; `"summary doesn't end with a full stop"`; `"summary longer than 79 characters"`; `"no blank line after the summary"` if there's a second line that isn't blank; and `"summary starts with 'This function'"` (in any case), since summaries are commands.

In [ ]:
def summary_problems(doc):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pep257():\n    \"PEP 257's first-line rules\"\n    assert summary_problems(\"Return the total.\") == []\n    assert summary_problems(\"Return the total.\\n\\n    More detail.\\n    \") == []\n    assert summary_problems(\"\") == [\"empty\"] and summary_problems(\"   \") == [\"empty\"]\n    assert summary_problems(\"\\n    Return the total.\\n    \") == [\"starts with a blank line\"]\n    assert summary_problems(\"This function returns the total\") == [\"summary doesn't end with a full stop\", \"summary starts with 'This function'\"]\n    assert summary_problems(\"Return the total.\\n    It adds tax too.\") == [\"no blank line after the summary\"]\n    assert summary_problems(\"Return \" + \"x\" * 80 + \".\") == [\"summary longer than 79 characters\"]")

### Bug hunt: An example that lies

The function is right, but its documentation isn't: the example's expected output shows an exclamation mark that `shout` doesn't add, so the doctest fails and this prints `TestResults(failed=1, attempted=1)`. Fix the **docstring** (not the function) so this prints `TestResults(failed=0, attempted=1)`.

In [ ]:
import doctest

def shout(text):
    """Return the text in capitals.

    >>> shout("hi")
    'HI!'
    """
    return text.upper()

runner = doctest.DocTestRunner()
for test in doctest.DocTestFinder().find(shout, globs={"shout": shout}):
    runner.run(test, out=lambda s: None)
print(runner.summarize(verbose=False))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"documentation that matches\"\n    expect_output(\"TestResults(failed=0, attempted=1)\")\n    assert \"return text.upper()\" in source")

### Bug hunt: A docstring that isn't one

The description comes **after** the first statement, so Python treats it as a plain string expression, not a docstring, and this prints `None`. Make it the docstring so this prints `Return the price with VAT added.`

In [ ]:
def with_vat(price):
    rate = 0.2
    """Return the price with VAT added."""
    return price * (1 + rate)

print(with_vat.__doc__)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the first statement\"\n    expect_output(\"Return the price with VAT added.\")")

### Bug hunt: Help that didn't inherit

`Square.area` overrides `Shape.area` without a docstring of its own, so the raw `__doc__` of a square's `area` method is `None`, and the help prints `None`. `inspect.getdoc` falls back to the docstring of the method it overrides, so use it and this prints `Return the area, in square metres.`

In [ ]:
import inspect

class Shape:
    def area(self):
        """Return the area, in square metres."""
        raise NotImplementedError

class Square(Shape):
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2

print(Square(2).area.__doc__)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"inherited documentation\"\n    expect_output(\"Return the area, in square metres.\")")

### Puzzle: A Google-style docstring

Arrange the lines of a function with a Google-style docstring. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # True or False: a string after the first statement of a function becomes its docstring
q = ___   # the doctest output line expected for: >>> 1 + 1
r = ___   # the pytest option that runs doctests in modules
s = ___   # the inspect function that returns a cleaned docstring
t = ___   # the docstring style with sections "Args:", "Returns:" and "Raises:": "Google" or "NumPy"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is False and q == \"2\" and r == \"--doctest-modules\", \"p, q and r\"\n    assert s == \"getdoc\" and t == \"Google\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: documentation that checks itself

Write `check_docs(source)`: run several documentation checks on a module's **source** (parse it with `ast`, and also execute it into a fresh namespace for the doctests) and return a sorted list of `(name, problem)` pairs, for each public top-level function:
- `"no docstring"` (and no other checks for that function);
- `"undocumented argument: NAME"` and `"no such argument: NAME"`, comparing a Google-style `Args:` section with the parameters (skip this check if the function has no parameters and no `Args:` section);
- `"no Returns section"` if the function has a `return` statement with a value (anywhere in its own body) but no `Returns:` section;
- `"undocumented exception: NAME"` for each exception class it raises directly (`raise Name(...)` or `raise Name`) that isn't in its `Raises:` section;
- `"failing example"` if any of its doctests fail (run with the executed module as the globals).
Use `parse_docstring` (given).

In [ ]:
import ast, doctest, inspect, re

HEADINGS = {"Args:": "args", "Arguments:": "args", "Returns:": "returns", "Raises:": "raises", "Examples:": "examples"}

def parse_docstring(doc):
    lines = inspect.cleandoc(doc).splitlines()
    result = {"summary": lines[0].strip() if lines else "", "args": {}, "returns": None, "raises": {}}
    section, returns, last = None, [], None
    for line in lines[1:]:
        stripped = line.strip()
        if stripped in HEADINGS:
            section, last = HEADINGS[stripped], None
            continue
        if section in ("args", "raises") and stripped:
            m = re.match(r"(\w+)\s*(?:\([^)]*\))?:\s*(.*)", stripped)
            if m and len(line) - len(line.lstrip()) <= 4:
                last = m.group(1)
                result[section][last] = m.group(2).strip()
            elif last is not None:
                result[section][last] += " " + stripped
        elif section == "returns" and stripped:
            returns.append(stripped)
    result["returns"] = " ".join(returns) if returns else None
    return result

def check_docs(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_checks():\n    \"a module's documentation, checked\"\n    src = '''\ndef withdraw(balance, amount):\n    \"\"\"Take money out.\n\n    Args:\n        balance: The current balance.\n        amt: How much.\n\n    Raises:\n        ValueError: If there isn't enough.\n\n    >>> withdraw(10, 3)\n    7\n    \"\"\"\n    if amount > balance:\n        raise ValueError(\"not enough\")\n    if amount < 0:\n        raise TypeError\n    return balance - amount\n\ndef greet(name):\n    \"\"\"Return a greeting.\n\n    Args:\n        name: Who to greet.\n\n    Returns:\n        The greeting.\n\n    >>> greet(\"Ada\")\n    'Hello Ada'\n    \"\"\"\n    return f\"Hello, {name}\"\n\ndef nothing():\n    \"\"\"Do nothing.\"\"\"\n\ndef bare(x):\n    return x\n\ndef _private(x):\n    return x\n'''\n    assert check_docs(src) == [\n        (\"bare\", \"no docstring\"),\n        (\"greet\", \"failing example\"),\n        (\"withdraw\", \"no Returns section\"),\n        (\"withdraw\", \"no such argument: amt\"),\n        (\"withdraw\", \"undocumented argument: amount\"),\n        (\"withdraw\", \"undocumented exception: TypeError\"),\n    ]")

## Warm-up

The docstring **promises** that `average([])` returns `0`, and running its examples shows the code doesn't keep the promise (it raises `ZeroDivisionError`). Examples in documentation that are also tests can't quietly go wrong: either the code or the documentation gets fixed.

## Docstrings, properly

**PEP 257** sets the conventions:

- A docstring is the **first statement** in a module, function, class or method: a string literal, with triple double quotes (`"""`), even for one line. Anywhere else, a string is just an expression that's thrown away.
- The **first line** is a one-line **summary**, ending with a full stop, written as a command for functions ("Return the mean...", not "Returns..." or "This function..."). It should fit on one line and make sense on its own, because tools show it alone in lists and tooltips.
- For more, a **blank line**, then the details: what each argument means, what's returned, what's raised, side effects, and examples.
- Document **behaviour and intent** that the signature can't show: units, ranges, what `None` means, which exceptions to expect. Don't repeat type hints in prose; do explain what the values mean.

For the details, pick one structured style for the whole project, so tools can read it. The two common ones are **Google style**:

```python
def transfer(source, target, amount):
    """Move money between two accounts.

    Args:
        source: The account to take money from.
        target: The account to pay into.
        amount: How much to move, in pence. Must be positive.

    Returns:
        The new balance of the source account.

    Raises:
        ValueError: If amount isn't positive or the source can't cover it.
    """
```

and **NumPy style**, with underlined section headings (`Parameters`, `----------`), which suits long scientific APIs. Sphinx can read both.

`help(obj)` and `inspect.getdoc(obj)` show a docstring with its indentation cleaned up (`inspect.cleandoc` does the cleaning), and for a method that overrides another without a docstring of its own, they use the overridden method's; `obj.__doc__` is the raw text.

## Doctests

A **doctest** is an example written like an interactive session: a line starting `>>> ` (with `... ` for continuation lines), followed by the exact output Python would print. The `doctest` module finds them in docstrings and runs them:

- `python -m doctest module.py -v` on the command line, or `pytest --doctest-modules`, which runs them as part of the test suite;
- in code: `doctest.testmod()` for the current module, or `DocTestFinder` and `DocTestRunner` for finer control (the warm-up).

The output must match **exactly** (it's compared as text), which is both the strength and the weakness: write examples whose output is stable. Tips:

- Show results that print simply: numbers, short strings, lists, `True`. Sort sets and dicts first, or compare: `>>> sorted(tags)`.
- Exceptions are shown with a `Traceback (most recent call last):` line, then `...`, then the last line: `ValueError: amount must be positive`. doctest only compares the first and last lines.
- Options in comments loosen the comparison: `# doctest: +ELLIPSIS` lets `...` match anything, and `+NORMALIZE_WHITESPACE` ignores spacing differences.

Doctests are **documentation first**: a few clear examples that also run. Thorough edge-case testing belongs in the test suite.

## Reference from the code

Reference documentation (every public function, class and method, with its signature and description) should be **generated from the code**, so it can't disagree with it. Python makes the raw material easy to get:

- at run time, `inspect.signature(func)` gives the parameters with their defaults and annotations, and `inspect.getdoc(func)` the cleaned docstring;
- without running the code, `ast` gives the same from the source: `ast.get_docstring(node)` for any function, class or module node, and `ast.unparse(node.args)` for a parameter list.

Tools built on this: **pydoc** (in the standard library: `python -m pydoc -b` serves documentation for everything installed), **Sphinx** with its `autodoc` extension, and **mkdocstrings** for MkDocs.

The same information lets you **check** documentation: is every public name documented (**docstring coverage**, measured by tools such as `interrogate`)? Does every documented argument exist, and is every argument documented? Ruff's `D` rules (from pydocstyle) check PEP 257's formatting rules.

## Documentation sites and READMEs

Larger projects publish a **documentation site**, written as Markdown or reStructuredText files in a `docs/` folder next to the code ("docs as code": reviewed in pull requests, built in CI, versioned with releases):

- **Sphinx** (reStructuredText or Markdown, the standard for Python itself and many big projects) and **MkDocs** with the Material theme (Markdown, popular for its simplicity) turn the files, plus reference pages generated from docstrings, into a website.
- Sites such as Read the Docs build and host the documentation for each release automatically.
- Organise the site by the four kinds (Week 22 Day 5): a tutorial for newcomers, how-to guides for tasks, reference generated from the code, and explanations of the design.

Every project, however small, needs a **README** that answers, in order: what is this; how do I install it; a short example of using it; where's the documentation; how do I contribute or report a problem; what's the licence. Keep the example runnable (some projects doctest their README too). A `CONTRIBUTING.md` explains how to set up a development environment and run the tests, and the changelog (Week 73 Day 4) tells users what changed.

## Practice

## Quiz

## Daily challenge

## Before you go

Pick a module you wrote in this course and run today's checker on it: add the missing docstrings, with one doctest each. Tomorrow: **continuous integration**: running every check automatically, on every push, before anything is merged.